In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("ModuleofAppliedGeomorphologyfinal.pdf")
documents = loader.load()

print(len(documents))

Data cleanining 

In [ ]:
clean_docs = []

for doc in documents:
    text = doc.page_content.replace("\n", " ")
    text = text.strip()
    clean_docs.append(text)

Chunking

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.create_documents(clean_docs)

print(len(chunks))

Embedding Generation (Open Source)

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Store in Vector Database (FAISS)

In [ ]:
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_documents(
    documents=chunks,
    embedding=embedding_model
)

vector_store.save_local("faiss_index")

Retrieval

In [ ]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

query = "What is the main topic of the document?"

retrieved_docs = retriever.get_relevant_documents(query)

for doc in retrieved_docs:
    print(doc.page_content)

Pass Context to LLM (Prompt Construction)

In [ ]:
from langchain_community.llms import Ollama

llm = Ollama(model="tinyllama")

In [ ]:
# Build context from retrieved documents
context = ""

for doc in retrieved_docs:
    context = context + doc.page_content + "\n\n"

# Create prompt
prompt = "Answer the question based ONLY on the context below.\n\n"
prompt = prompt + "Context:\n"
prompt = prompt + context + "\n"
prompt = prompt + "Question:\n"
prompt = prompt + query + "\n\n"
prompt = prompt + "Answer:"

Generate Answer

In [ ]:
response = llm.invoke(prompt)

print(response)